In [1]:
def elo_update(team_elo, opponent_elo, location, score_a, score_b, competition, k_base=25):
    # 1. Résultat du match
    if int(score_a) > int(score_b):
        R = 1.0
    elif int(score_a) < int(score_b):
        R = 0.0
    else:
        R = 0.5

    # 2. Bonus/malus selon le lieu
    if location == "home":
        H = 60
    elif location == "away":
        H = -60
    else:
        H = 0

    # 3. Score attendu
    E = 1 / (1 + 10 ** ((opponent_elo - team_elo - H) / 400))

    # 4. Coefficient K selon compétition et écart
    diff = abs(int(score_a) - int(score_b))
    if competition in ["ChampionsCup", "ChallengeCup"]:
        K = k_base * (1.8 if diff > 15 else 1.2)
    else:
        K = k_base * (1.5 if diff > 15 else 1)

    # 5. Mise à jour Elo avec arrondi à l’entier
    delta = K * (R - E)
    new_team_elo = round(team_elo + delta)
    new_opponent_elo = round(opponent_elo - delta)

    return new_team_elo, new_opponent_elo


# Equipes

In [2]:
teams = {
    "Bordeaux-Begles": 1789,
    "Toulouse": 1752,
    "Leinster": 1709,
    "Northampton": 1701,
    "Bath": 1693,
    "Montpellier": 1677,
    "Leicester": 1668,
    "Glasgow": 1658,
    "Bulls": 1625,
    "La Rochelle": 1574,
    "Stade Francais": 1565,
    "Saracens": 1565,
    "Stormers": 1561,
    "Pau": 1557,
    "Exeter": 1552,
    "Toulon": 1542,
    "Clermont": 1540,
    "Racing 92": 1533,
    "Bristol": 1531,
    "Munster": 1527,
    "Ulster": 1498,
    "Connacht": 1487,
    "Sale": 1479,
    "Sharks": 1479,
    "Castres": 1478,
    "Harlequins": 1475,
    "Lions": 1462,
    "Trevise": 1453,
    "Cardiff": 1452,
    "Gloucester": 1442,
    "Lyon": 1440,
    "Bayonne": 1431,
    "Edimbourg": 1430,
    "Scarlets": 1369,
    "Ospreys": 1366,
    "Perpignan": 1348,
    "Dragons": 1242,
    "Cheetahs": 1238,
    "Zebre": 1216,
    "Black Lion": 1211,
    "Montauban": 1154,
    "Newcastle": 1144
}

# TOP 14

In [3]:
TOP14 = [

("Montauban","La Rochelle","home",15,71),
("Toulouse","Lyon","home",39,31),
("Perpignan","Castres","home",29,27),
("Stade Francais","Bayonne","home",38,21),
("Montpellier","Pau","home",26,18),
("Toulon","Bordeaux-Begles","home",27,22),
("Clermont","Racing 92","home",13,41),

               ]

In [4]:
Premiership = [ 

("Bristol","Bath","home",21,19),
("Saracens","Harlequins","home",26,12),
("Northampton","Gloucester","home",36,32),
("Newcastle","Sale","home",45,42),
("Leicester","Exeter","home",26,35),

               ]

# URC

In [5]:
URC = [

("Glasgow","Connacht","home",33,21),
("Bulls","Munster","home",45,14),
("Stormers","Cardiff","home",44,21),
("Leinster","Lions","home",59,10),

               ]

# Champions Cup

In [6]:

ChampionsCup = [







]

# Challenge Cup

In [7]:

ChallengeCup = [ 





               ]

In [8]:
import csv
import pandas as pd

initial_ranking = {team: rank for rank, team in enumerate(sorted(teams, key=teams.get, reverse=True))}

# --- Traitement des matchs ---
for match_list, competition in zip(
    [TOP14, Premiership, URC, ChampionsCup, ChallengeCup],
    ["TOP14", "Premiership", "URC", "ChampionsCup", "ChallengeCup"]
):
    for match in match_list:
        team_a, team_b, location, score_a, score_b = match
        team_points, opponent_points = elo_update(
            teams[team_a],
            teams[team_b],
            location,
            score_a,
            score_b,
            competition
        )
        teams[team_a] = team_points
        teams[team_b] = opponent_points

# --- Export du classement ---
with open("rankings.csv", "w", newline="") as csvfile:
    fieldnames = ["Team", "Points"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for team in sorted(teams, key=teams.get, reverse=True):
        writer.writerow({"Team": team, "Points": round(teams[team], 2)})

# --- Comparaison initial/final ---
df = pd.read_csv('rankings.csv')
final_ranking = {team: rank for rank, team in enumerate(df["Team"])}
ranking_changes = {team: initial_ranking[team] - final_ranking[team] for team in teams}

gained_places = {team: change for team, change in ranking_changes.items() if change > 0}
lost_places = {team: change for team, change in ranking_changes.items() if change < 0}

# --- Affichage final ---
print("-------------CLIMBERS-------------\n")
for team, change in sorted(gained_places.items(), key=lambda item: item[1], reverse=True):
    print(f"{team}: +{change}")

print("\n-------------LOSERS-------------\n")
for team, change in sorted(lost_places.items(), key=lambda item: item[1]):
    print(f"{team}: -{abs(change)}")

df

-------------CLIMBERS-------------

Racing 92: +2
Harlequins: +2
Montpellier: +1
Glasgow: +1
Exeter: +1
Toulon: +1
Bristol: +1
Sharks: +1
Edimbourg: +1
Newcastle: +1

-------------LOSERS-------------

Pau: -3
Sale: -3
Clermont: -2
Bath: -1
Leicester: -1
Bayonne: -1
Montauban: -1


,Team,Points
0,Bordeaux-Begles,1770
1,Toulouse,1755
2,Leinster,1714
3,Northampton,1704
4,Montpellier,1684
5,Bath,1677
6,Glasgow,1663
7,Leicester,1650
8,Bulls,1636
9,La Rochelle,1578
